In [1]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list, add_messages]

graph_builder = StateGraph(State)

In [2]:
# LLM 모델 초기화.
from langchain.chat_models import init_chat_model
llm = init_chat_model("openai:gpt-4.1")

In [3]:
# 다음과 같이 챗봇을 정의하는 함수를 작성함.
# 이 함수는 상태 객체 State 를 받아 그 안에 메시지 리스트를 GPT 모델에 전달하고 응답 메시지를 반환함.

# 노드 정의 : 상태를 받아 새로운 메시지를 생성해 줌.
def chatbot(state:State):
    return {"messages": [llm.invoke(state["messages"])]}

# 이 함수는 "messages"라는 키를 가진 딕셔너리를 반환하며, 앞서 정의한 state 구조에 따라 이 메시지는 기존 상태의 메시지 리스트에 자동으로 추가됨. 

In [4]:
# 마지막으로 함수를 StateGraph에 노드로 등록함.
# 첫번째 인자는 노드의 이름이며, 두번째 인자는 노드 함수임.
graph_builder.add_node("chatbot", chatbot)

# 이러한 방식으로 각 노드는 자신의 작업만을 담당하면서 상태를 주고 받으며 자연스럽게 전체 워크 플로우를 구성할 수 있음.

In [5]:
# 엣지 정의 : 그래프의 시작점에서 chatbot 노드로 흐름을 연결함.
# add_edge 함수를 사용하여 엣지를 정의함. 첫번째 인자는 시작노드이고 두번째 인자는 이동할 대상 노드임.
# 단순한 직선 흐름 뿐 아니라 분기 또는 반복되는 흐름도 표현할 수 있어 복잡한 시나리오를 유연하게 처리할 수 있음.
# 특히 그래프의 시작점을 설정할 때는 START 상수를 사용함.
graph_builder.add_edge(START, "chatbot")

# 위 코드는 그래프가 실행될 때 가장 먼저 chatbot 노드로 흐름이 전달되도록 지정함.
# 이처럼 엣지를 통해 워크플로우의 방향과 순서를 명확히 설정할 수 있으며, 이후 더 복잡한 흐름도 조건 분기를 통해 유연하게 확장할 수 있음.

In [6]:
# 그래프 컴파일 및 시각화
# 지금까지 정의한 상태, 노드, 엣지를 바탕으로 실제 실행 가능한 그래프를 만들려면 컴파일해야 함.
# 컴파일이란 정의한 그래프 구조를 실행 가능한 워크플로우 객체로 변환하는 과정을 의미함.
# graph_builder.compile() 을 호출하면 우리가 정의한 작업 흐름이 CompileGraph라는 형태로 정리되어 이후 실제 상태를 넣고 실행할 수 있게 됨.

# 그래프를 컴파일하여 실행 가능한 형태로 반환함.
graph = graph_builder.compile()

In [7]:
# 컴파일된 그래프는 내부적으로 상태 전이 구조와 노드 간 흐름 정보를 모두 포함하고 있으며, 이후 챗봇을 실행할 때 이 구조를 바탕으로 동작하게 됨.

# 또한 랭그래프는 이 그래프 구조를 시각화 하는 기능도 제공함.
# get_graph() 메서드와 함께 draw_ascii(), draw_mermaid_png() 같은 메서드를 활용하면 워크플로우를 그림으로 확인할 수 있음.

# 다음 예시는 머메이드(mermaid)형식의 PNG 이미지로 그래프를 시각화하는 코드임 
# 머메이드 형식이란 "코드로 그리는 다이어그램 언어" 
# 마크다운 문서내에서 복잡한 이미지 파일 없이, 오직 텍스트만으로 플로우 챠트, 시퀀스 다이어그램, 간트 차트 등을 쉽게 빠르게 그려낼수 있도록 도와주는 오픈소스 도구 인터프리터이자 문법임.

# 그래프 구조를 시각적으로 출력함 (선택 사항.)
#from IPython.display import Image, display

#try:
#    display(Image(graph.get_graph().draw_mermaid_png()))
#except Exception:
    # 시각화를 위해 별도 라이브러리가 필요하며, 없을 경우 무시해도 됨.
#    pass

In [ ]:
from langchain_core.messages import HumanMessage

# 만약 챗봇을 실제로 실행해 보고 싶다면 다음과 같이 사용자 입력값을 받아 스트리밍 방식으로 응답을 출력할 수 있음.
#   . stream_graph_updates() 함수는 사용자 입력을 랭그래프로 전달하고, 생성된 응답을 순차적으로 출력함.
#   . while True 루프를 사용해 사용자가 종료 명령을 입력할 때까지 챗봇과 계속 대화할 수 있음.
#   . 예제에서는 quit, exit, p 중 하나를 입력하면 대화를 종료하도록 설정했음.
def stream_graph_updates(user_input: str):
    # graph.stream() 은 랭그래프를 단계별로 실행하며 중간 결과를 제공함.
    for event in graph.stream({
        #"messages": [{"role": "user", "content": user_input}]
        "messages": [HumanMessage(content=user_input)]
    }):
        # 각 이벤트에는 노드의 실행 결과가 담겨 있음.
        for value in event.values():
            # 가장 마지막 메시지를 꺼내어 출력함.
            print("Assistant:", value["messages"][-1].content)

# 챗봇을 반복적으로 사용할 수 있도록 무한 루프를 설정함.
while True:
    # 사용자에게 입력을 받음.
    user_input = input("User: ").strip()

    # 입력값이 비어있다면 대기하지 않고 스킵 (Jupyter 먹통 방지)
    if not user_input:
        continue

    # 사용자가 종료 명령을 입력하면 프로그램을 종료함.
    if user_input.lower() in ["quit", "exit", "q"]:
        print("Goodbye!")
        break

    # 입력한 내용을 바탕으로 챗봇의 응답을 실행함.
    stream_graph_updates(user_input)